# M³Diff — demo checkpoint B (LoRA + SCST)

**Dùng web:** bật GPU A100, chạy **1 → 2 → 3 → 6 → 7**. Ô 4–5 tùy chọn.

- Ô 2 tự tìm đúng B128 trên Drive và xuất adapter dùng cho demo; không huấn luyện thêm.
- Chưa có `MyDrive/M3Diff_Storage/demo_scst_B128/` là bình thường trước lần xuất đầu tiên. Ô 2 tự tạo từ checkpoint trong `MDP_Projector_v1/experiments/scst_lora_20261031/`; không cần tạo thư mục bằng tay.
- Ô 3 nạp **một model B**, chọn hai ảnh để kiểm tra; các lượt sau dùng lại model.
- Laptop: mở **CHAY_DEMO.cmd**, nhập file kết nối do ô 7 tải xuống.
- Chia sẻ notebook và Drive `M3Diff_Storage/` cho nhóm. Thành viên mount Drive để truy cập đúng `MyDrive/M3Diff_Storage/`, gồm model gốc và thí nghiệm B128. Ô 2 tự chuẩn bị adapter khi cần; không phải gửi thêm checkpoint riêng. Lần tạo adapter đầu tiên cần quyền ghi vào thư mục Drive này.
- Cấu hình B khớp đường đánh giá full1570: BF16, MDP/projector FP32, cùng prompt và cách sinh câu.
- File kết nối chứa mật khẩu; không push lên GitHub. Sau khi mất runtime, chạy lại 1 → 2 → 3 → 6 → 7.

Tài liệu phương pháp/kết quả: `CAI_TIEN_SCST_VA_KET_QUA.md`; hướng dẫn chia sẻ Drive: `CHECKPOINT_VA_CHIA_SE.md` trong thư mục demo.


In [ ]:
#@title 1. Chuẩn bị demo B
import sys, subprocess, importlib.util, json, base64, hashlib, io, zipfile, os
from pathlib import Path, PurePosixPath
WORK=Path('/content/m3diff_demo_B');WORK.mkdir(parents=True,exist_ok=True)
SOURCE=Path('/content/m3diff_demo_scst_source');SOURCE.mkdir(parents=True,exist_ok=True)
VENV=Path('/content/m3diff-demo-venv');VENV_PY=VENV/'bin/python'
MODEL_PATH=Path('/content/m3diff_train_checkpoint')
SUPPORT_FILES = {'scst_artifacts.py': '"""Export and verify the exact B128 adapter; CPU-only until preparation imports Torch.\n\nThe training checkpoint is our own pinned, SHA-verified artifact. Never load an\narbitrary .pt received from a third party with weights_only=False.\n"""\nfrom pathlib import Path\nimport argparse\nimport hashlib\nimport json\nimport os\nimport shutil\nimport zipfile\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef identity(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n\n\ndef sha256(path):\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(8 * 1024**2), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    tmp = path.with_name(path.name + \'.tmp\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2) + \'\\n\', encoding=\'utf-8\')\n    os.replace(tmp, path)\n\n\ndef validate_commit(run, release):\n    run = Path(run)\n    if identity(read(run / \'protocol.json\')) != release[\'parent_protocol\']:\n        raise ValueError(\'Sai protocol huấn luyện B128; không tự dùng checkpoint khác.\')\n    pointer = read(run / \'checkpoints/B/commit.json\')\n    if (pointer.get(\'arm\') != \'B\' or pointer.get(\'step\') != release[\'step\']\n            or pointer.get(\'contract\') != release[\'parent_protocol\']\n            or type(pointer.get(\'slot\')) is not int or pointer[\'slot\'] not in (0, 1)\n            or pointer.get(\'sha256\') != release[\'checkpoint_sha256\']):\n        raise ValueError(\'Cần đúng checkpoint B bước128 đã đánh giá, theo commit.json.\')\n    expected = {\'protocol\': release[\'parent_protocol\'], \'weights\': release[\'checkpoint_sha256\'], \'step\': release[\'step\']}\n    if read(run / \'weights_B.json\') != expected:\n        raise ValueError(\'weights_B.json không khớp kết quả full1570.\')\n    path = run / \'checkpoints/B\' / f"slot{pointer[\'slot\']}.pt"\n    if sha256(path) != release[\'checkpoint_sha256\']:\n        raise ValueError(\'Checkpoint B hỏng hoặc sai SHA256; chưa nạp trọng số.\')\n    return path\n\n\ndef verify_adapter(folder, release):\n    folder = Path(folder)\n    manifest = read(folder / \'adapter_manifest.json\')\n    if manifest.get(\'release\') != release or manifest.get(\'format\') != \'m3diff_lora_tensors_v1\':\n        raise ValueError(\'Gói adapter không thuộc bản B128 đã chọn.\')\n    if sha256(folder / \'adapter_model.pt\') != manifest.get(\'adapter_sha256\'):\n        raise ValueError(\'Adapter bị hỏng hoặc thiếu dữ liệu.\')\n    return manifest\n\n\ndef export_adapter(run, destination, release):\n    destination = Path(destination)\n    if (destination / \'adapter_manifest.json\').exists():\n        verify_adapter(destination, release)\n        return destination\n    path = validate_commit(run, release)  # Verify before trusted pickle deserialization.\n    import torch\n    state = torch.load(path, map_location=\'cpu\', weights_only=False)\n    if (state.get(\'arm\'), state.get(\'contract\'), state.get(\'step\')) != (\'B\', release[\'parent_protocol\'], release[\'step\']):\n        raise ValueError(\'Nội dung checkpoint không khớp commit.\')\n    weights = state.get(\'adapter\')\n    if not isinstance(weights, dict) or not weights:\n        raise ValueError(\'Checkpoint không có adapter.\')\n    for key, value in weights.items():\n        if (\'lora_\' not in key or not torch.is_tensor(value) or value.ndim != 2\n                or not value.is_floating_point() or not torch.isfinite(value).all()):\n            raise ValueError(\'Trọng số LoRA không hợp lệ: \' + key)\n    tensors = {key: value.detach().cpu().contiguous().clone() for key, value in weights.items()}\n    destination.mkdir(parents=True, exist_ok=True)\n    target = destination / \'adapter_model.pt\'\n    temporary = target.with_suffix(\'.tmp\')\n    torch.save(tensors, temporary)\n    loaded = torch.load(temporary, map_location=\'cpu\', weights_only=True)\n    if set(loaded) != set(tensors) or any(not torch.equal(loaded[k], v) for k, v in tensors.items()):\n        raise ValueError(\'Xuất/nạp lại adapter không khớp.\')\n    os.replace(temporary, target)\n    atomic_json(destination / \'adapter_manifest.json\', {\n        \'format\': \'m3diff_lora_tensors_v1\', \'release\': release,\n        \'adapter_sha256\': sha256(target), \'size_bytes\': target.stat().st_size,\n        \'tensor_shapes\': {k: list(v.shape) for k, v in tensors.items()},\n        \'tensor_dtypes\': {k: str(v.dtype) for k, v in tensors.items()},\n        \'note\': \'Inference-only adapter; requires the original M3Diff checkpoint. No optimizer or RNG.\',\n    })\n    verify_adapter(destination, release)\n    return destination\n\n\ndef copy_checked(source, target, digest):\n    source, target = Path(source), Path(target)\n    if target.is_file() and sha256(target) == digest:\n        return\n    if not source.is_file():\n        raise FileNotFoundError(\'Thiếu model gốc: \' + str(source))\n    target.parent.mkdir(parents=True, exist_ok=True)\n    temporary = target.with_name(target.name + \'.incomplete\')\n    with source.open(\'rb\') as src, temporary.open(\'wb\') as dst:\n        shutil.copyfileobj(src, dst, 8 * 1024**2)\n    if sha256(temporary) != digest:\n        raise ValueError(\'Sai checksum: \' + str(source))\n    os.replace(temporary, target)\n\n\ndef prepare(storage, release_path, protocol_path):\n    from huggingface_hub import snapshot_download\n    storage = Path(storage)\n    release = read(release_path)\n    run = storage / \'MDP_Projector_v1/experiments/scst_lora_20261031\'\n    package = storage / \'demo_scst_B128\'\n    if not (package / \'adapter_manifest.json\').exists():\n        print(\'Xuất adapter B128 từ checkpoint đã đánh giá (CPU).\', flush=True)\n        export_adapter(run, package, release)\n    verify_adapter(package, release)\n    model = Path(\'/content/m3diff_train_checkpoint\')\n    for name, digest in release[\'base_files\'].items():\n        print(\'Kiểm tra model gốc:\', name, flush=True)\n        copy_checked(storage / \'M3Diff_checkpoint\' / name, model / name, digest)\n    v = release[\'vision\']\n    vision = Path(snapshot_download(v[\'repo\'], revision=v[\'revision\'],\n        allow_patterns=list(release[\'vision_hashes\']), cache_dir=\'/content/hf_cache\'))\n    for name, digest in release[\'vision_hashes\'].items():\n        if sha256(vision / name) != digest:\n            raise ValueError(\'Sai SigLIP: \' + name)\n    atomic_json(protocol_path, {\n        \'model_variant\': \'B_SCST_128\', \'adapter_path\': str(package),\n        \'release_path\': str(Path(release_path).resolve()), \'vision_path\': str(vision),\n        \'prompt\': release[\'question\'], \'torch_dtype\': \'bfloat16\',\n        \'generation\': release[\'generation\'], \'mdp_projector_dtype\': \'float32\',\n        \'parent_protocol\': release[\'parent_protocol\'],\n    })\n    print(\'B128 READY:\', package, \'| Model:\', model, flush=True)\n\n\ndef write_export_zip(folder, target, release):\n    folder, target = Path(folder), Path(target)\n    verify_adapter(folder, release)\n    with zipfile.ZipFile(target, \'w\', compression=zipfile.ZIP_DEFLATED) as z:\n        for name in (\'adapter_model.pt\', \'adapter_manifest.json\'):\n            z.write(folder / name, \'demo_scst_B128/\' + name)\n    return target\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'command\', choices=[\'prepare\', \'export\'])\n    parser.add_argument(\'--storage\', default=\'/content/drive/MyDrive/M3Diff_Storage\')\n    parser.add_argument(\'--release\', required=True)\n    parser.add_argument(\'--protocol\')\n    parser.add_argument(\'--zip\')\n    args = parser.parse_args()\n    if args.command == \'prepare\':\n        if not args.protocol:\n            parser.error(\'--protocol required\')\n        prepare(args.storage, args.release, args.protocol)\n    else:\n        if not args.zip:\n            parser.error(\'--zip required\')\n        storage = Path(args.storage)\n        release = read(args.release)\n        folder = export_adapter(storage / \'MDP_Projector_v1/experiments/scst_lora_20261031\', storage / \'demo_scst_B128\', release)\n        print(write_export_zip(folder, args.zip, release))\n', 'scst_demo_runner.py': '"""Persistent B128 inference matching the full1570 evaluation path."""\nfrom copy import deepcopy\nfrom datetime import datetime, timezone\nimport importlib\nfrom importlib import metadata\nfrom pathlib import Path\nimport random\nimport threading\nfrom time import perf_counter\nimport uuid\n\nfrom PIL import Image\nfrom scst_artifacts import read, sha256, identity, verify_adapter\nfrom inference_tokens import token_layout\n\n\ndef validate_protocol(protocol, release):\n    if (protocol.get(\'model_variant\') != \'B_SCST_128\'\n            or protocol.get(\'prompt\') != release[\'question\']\n            or protocol.get(\'generation\') != release[\'generation\']\n            or protocol.get(\'torch_dtype\') != \'bfloat16\'\n            or protocol.get(\'mdp_projector_dtype\') != \'float32\'\n            or protocol.get(\'parent_protocol\') != release[\'parent_protocol\']):\n        raise ValueError(\'Cấu hình demo khác bản B đã đánh giá full1570.\')\n\n\ndef restore_adapter(model, state, get_state, set_state, torch):\n    expected = get_state(model)\n    if set(state) != set(expected) or any(state[k].shape != expected[k].shape for k in expected):\n        raise ValueError(\'Sai keys hoặc kích thước adapter; không chạy với LoRA thiếu trọng số.\')\n    if any(not torch.isfinite(v).all() for v in state.values()):\n        raise ValueError(\'Adapter chứa NaN/Inf.\')\n    set_state(model, state)\n    actual = get_state(model)\n    if any(not torch.equal(actual[k].detach().cpu(), v.cpu()) for k, v in state.items()):\n        raise ValueError(\'Nạp adapter không khớp từng tensor.\')\n\n\nclass SCSTRunner:\n    def __init__(self, model_path, protocol):\n        self._protocol = deepcopy(protocol)\n        release = read(protocol[\'release_path\'])\n        validate_protocol(protocol, release)\n        self.model_path = Path(model_path).expanduser().absolute()\n        manifest = verify_adapter(protocol[\'adapter_path\'], release)\n        # Fail on a wrong base/vision before allocating a model on GPU.\n        for name, digest in release[\'base_files\'].items():\n            if sha256(self.model_path / name) != digest:\n                raise ValueError(\'Model gốc không khớp: \' + name)\n        vision_path = Path(protocol[\'vision_path\'])\n        for name, digest in release[\'vision_hashes\'].items():\n            if sha256(vision_path / name) != digest:\n                raise ValueError(\'SigLIP không khớp: \' + name)\n        import llava\n        source_root = Path(llava.__file__).resolve().parent.parent\n        for name, digest in release[\'source_files\'].items():\n            if sha256(source_root / name) != digest:\n                raise ValueError(\'Source demo khác bản đã đánh giá: \' + name)\n        torch = importlib.import_module(\'torch\')\n        self._torch = torch\n        if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():\n            raise RuntimeError(\'Demo B cần GPU hỗ trợ BF16; chọn A100 như lượt đánh giá.\')\n        from transformers import AutoTokenizer, GenerationConfig\n        from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict\n        from llava.model.language_model.llava_qwen import LlavaQwenForCausalLM, LlavaQwenConfig\n        from llava.mm_utils import process_images\n        self._process_images = process_images\n        random.seed(release[\'seed\']); torch.manual_seed(release[\'seed\']); torch.cuda.manual_seed_all(release[\'seed\'])\n        torch.backends.cuda.matmul.allow_tf32 = False\n        config = LlavaQwenConfig.from_pretrained(str(self.model_path), local_files_only=True)\n        if (config.mm_resampler_type != \'mdp\' or config.image_aspect_ratio != \'pad\'\n                or getattr(config, \'attention_dropout\', 0.) != 0.):\n            raise ValueError(\'Kiến trúc model không khớp.\')\n        config.unfreeze_mm_vision_tower = True  # Match evaluator loading, then freeze all parameters.\n        config.mm_vision_tower = str(vision_path)\n        config.use_cache = False\n        config.tokenizer_model_max_length = 4096\n        started = perf_counter()\n        base, loading_info = LlavaQwenForCausalLM.from_pretrained(str(self.model_path), config=config,\n            local_files_only=True, torch_dtype=torch.bfloat16, device_map={\'\': 0},\n            attn_implementation=\'sdpa\', low_cpu_mem_usage=True, output_loading_info=True)\n        if any(loading_info.get(k) for k in [\'missing_keys\', \'unexpected_keys\', \'mismatched_keys\', \'error_msgs\']):\n            raise ValueError(\'Model loading mismatch: \' + str(loading_info))\n        vision = base.get_vision_tower()\n        if not vision.is_loaded:\n            raise ValueError(\'Vision tower chưa được nạp.\')\n        vision.to(device=\'cuda\', dtype=torch.bfloat16)\n        inner = base.get_model()\n        inner.vision_resampler.float(); inner.mm_projector.float(); base.requires_grad_(False)\n        targets = [n for n, m in base.named_modules() if n.startswith(\'model.layers.\')\n            and n.endswith(tuple(release[\'lora\'][\'target_suffixes\'])) and isinstance(m, torch.nn.Linear)]\n        if len(targets) != 2 * config.num_hidden_layers:\n            raise ValueError(\'Sai số lớp q_proj/v_proj cho LoRA.\')\n        lc = {k: v for k, v in release[\'lora\'].items() if k != \'target_suffixes\'}\n        self.model = get_peft_model(base, LoraConfig(target_modules=targets, **lc))\n        for name, p in self.model.named_parameters():\n            if p.requires_grad:\n                if \'lora_\' not in name:\n                    raise ValueError(\'Có tham số ngoài LoRA chưa được đóng băng.\')\n                p.data = p.data.float()\n        state = torch.load(Path(protocol[\'adapter_path\']) / \'adapter_model.pt\', map_location=\'cpu\', weights_only=True)\n        restore_adapter(self.model, state, get_peft_model_state_dict, set_peft_model_state_dict, torch)\n        del state\n        self.model.requires_grad_(False); self.model.eval()\n        self.tokenizer = AutoTokenizer.from_pretrained(str(self.model_path), use_fast=False, local_files_only=True)\n        if self.tokenizer.convert_tokens_to_ids(\'<|im_end|>\') != 151645 or self.tokenizer.bos_token_id is not None:\n            raise ValueError(\'Tokenizer không khớp bản đánh giá.\')\n        self.generation_config = GenerationConfig(**release[\'generation\'])\n        self.image_processor = vision.image_processor\n        self.config = config\n        torch.cuda.synchronize()\n        self.load_seconds = perf_counter() - started\n        self.model_load_count = 1\n        self.successful_requests = 0\n        self.model_instance_id = uuid.uuid4().hex\n        self._lock = threading.Lock()\n        self.runtime_manifest = {\n            \'created_at_utc\': datetime.now(timezone.utc).isoformat(), \'model_variant\': \'B_SCST_128\',\n            \'adapter_manifest\': manifest, \'loading_info\': loading_info,\n            \'source_sha256\': release[\'source_sha256\'], \'base_hashes_verified\': True,\n            \'vision_hashes_verified\': True, \'gpu\': torch.cuda.get_device_name(0),\n            \'precision\': \'BF16 autocast; MDP/projector FP32; LoRA FP32\',\n            \'package_versions\': {n: metadata.version(n) for n in [\'torch\', \'transformers\', \'peft\']},\n            \'generation\': self.generation_config.to_dict(),\n        }\n        self.protocol_id = identity({\'protocol\': self._protocol, \'adapter_sha256\': manifest[\'adapter_sha256\'],\n            \'source_sha256\': release[\'source_sha256\']})\n\n    @property\n    def protocol(self):\n        return deepcopy(self._protocol)\n\n    def compare(self, before, after):\n        if not isinstance(before, Image.Image) or not isinstance(after, Image.Image):\n            raise TypeError(\'Cần hai ảnh PIL, theo thứ tự trước/sau.\')\n        torch = self._torch\n        with self._lock:\n            torch.cuda.synchronize(); started = perf_counter()\n            row = {\'conversations\': [{\'from\': \'human\', \'value\': self._protocol[\'prompt\']}]}\n            prompt, _ = token_layout(self.tokenizer, row, False)\n            pixels = [self._process_images([im.convert(\'RGB\')], self.image_processor, self.config)\n                .to(device=\'cuda\', dtype=torch.bfloat16) for im in (before, after)]\n            pixels = [torch.cat(pixels, dim=0)]\n            self.model.eval()\n            with torch.inference_mode(), torch.autocast(\'cuda\', dtype=torch.bfloat16):\n                output = self.model.generate(inputs=torch.tensor([prompt], device=\'cuda\'),\n                    attention_mask=torch.ones((1, len(prompt)), dtype=torch.long, device=\'cuda\'),\n                    images=pixels, generation_config=self.generation_config)\n            ids = output[0].detach().cpu().tolist()\n            if not ids or len(ids) > 256:\n                raise ValueError(\'Unexpected generated token layout\')\n            caption = self.tokenizer.decode(ids, skip_special_tokens=True).strip()\n            if not caption:\n                raise RuntimeError(\'Model trả caption rỗng.\')\n            torch.cuda.synchronize(); self.successful_requests += 1\n            return {\'caption\': caption, \'elapsed_seconds\': round(perf_counter() - started, 4),\n                \'protocol_id\': self.protocol_id, \'model_instance_id\': self.model_instance_id,\n                \'model_load_count\': self.model_load_count, \'successful_requests\': self.successful_requests,\n                \'model_variant\': \'B_SCST_128\'}\n', 'scst_release.json': '{\n  "model_variant": "B_SCST_128",\n  "step": 128,\n  "parent_protocol": "628f3f26f7cf66d55dbe0bb70b8d99ababbf243df72979de225336293a330ac6",\n  "checkpoint_sha256": "20d99f77346716fef35d5b369f6eee83458e8c854566d7bd3a557e5b2ba53a83",\n  "base_files": {\n    "added_tokens.json": "f86f2a952b4888d195b8d77e3d56ad96864ecc696cca2155d0d4ac933fcfd55f",\n    "config.json": "cd0608586cce1eb0af4f5ea61f873598fff45920a3b450609a17155f96eaaa95",\n    "generation_config.json": "e4eea1ce96e5935378f4d7f843c22b942998d59a8162abcc8891392afd98242c",\n    "merges.txt": "8831e4f1a044471340f7c0a83d7bd71306a5b867e95fd870f74d0c5308a904d5",\n    "model-00001-of-00004.safetensors": "267776d8371f377c158c759b7943237ee2dff7f349cf154b9972732e2f30f616",\n    "model-00002-of-00004.safetensors": "bcb22fc18202fc6bb87bd282db14430a3d5389e5f63242486c522fb185815d28",\n    "model-00003-of-00004.safetensors": "47262a139660a71769c9a7460c561c46dad36041a4cff7214ebb1f5f7227ad63",\n    "model-00004-of-00004.safetensors": "38849a8c36fc9bf6e85f149aa117adc38f8ba6ccab0a423fa1a79a3cc33ed81b",\n    "model.safetensors.index.json": "89817bb6a26ec95ce8382231a597b534ddc48b73619579566c670310cd1bfce5",\n    "special_tokens_map.json": "f4f79e08d97f4d1c87f8d89264f525c8789da3b73b3bb55d1e12f692f41a7b1b",\n    "tokenizer_config.json": "3ae8d5ad312360a6efd9b24299ca74b0f8342b41fe82799130a958c7000a92ff",\n    "vocab.json": "87a257b04b17642a0688c98cd1df89c398bda4fee532d6f88b38a659ecb4ac8d"\n  },\n  "vision": {\n    "repo": "google/siglip-so400m-patch14-384",\n    "revision": "9fdffc58afc957d1a03a25b10dba0329ab15c2a3"\n  },\n  "vision_hashes": {\n    "config.json": "adc04928d8fd19a61822584fe0cf2e813e5ebac17f3e49fb1ea096860ae6457b",\n    "model.safetensors": "ea2abad2b7f8a9c1aa5e49a244d5d57ffa71c56f720c94bc5d240ef4d6e1d94a",\n    "preprocessor_config.json": "f59da2f87c3cd079bd4f8f3037e81b277c60c498e279a8020331f67a5a3157e8",\n    "special_tokens_map.json": "2b6a1ff67a27e0df9ac0c7d93250fc0d87431c7b366b3d5669217104f9088a26",\n    "tokenizer.json": "c6e405cb7c670d56636a9402c81023a55bc6c3c53d89cf02b92f5c5005bfe920",\n    "tokenizer_config.json": "d6423dae508cc3a129d22ea443841c111832a1a73125b8f25ea8736951698bcb"\n  },\n  "source_sha256": "8a0c3e930942dcc2ea0d6fa69fc9a7226d47362e127583313b350e8d2961dffa",\n  "source_files": {\n    "llava/__init__.py": "e75fe3008976c75c35910b5460304d853d7a041b5bc573bc2d2cbcaf58e094d0",\n    "llava/constants.py": "680b1847d1566ebf6fc1dadd5e1720d7239f9f13d140b8521e4eb193c89e7fca",\n    "llava/conversation.py": "68021aa878befab9a91bdd30f6899bdb782818d08ea6479125237b5c7a5f42fd",\n    "llava/eval/model_vqa_qwen_CLEVR_Change.py": "d7056e000562da836f0d65f5e0e369e02ee505f1eb3993abf17a03436aff9d9d",\n    "llava/eval/model_vqa_qwen_CLEVR_DC.py": "7d287c617b50212dfd67a5ae9c994ecfc611391cd55f83146578b25c5f89b48d",\n    "llava/eval/model_vqa_qwen_IEdit.py": "aaddc7ebe043fbfe90d3c0ebc56c6918a477b20e502bebf0fe0df75606d90042",\n    "llava/eval/model_vqa_qwen_OmniDiff_Real.py": "13f20775066ab9d773ecac44f50f443ca72998e350bd3ee09d56f28ebf67bf22",\n    "llava/eval/model_vqa_qwen_OmniDiff_Render.py": "0c91dc5d3e317697b9babb8de72167b158847a1a3804210b50bbc1bab9312d7b",\n    "llava/eval/model_vqa_qwen_Spot_the_Diff.py": "70ee6c663f77fd995d9ca6899590176abd32a34427657ec64795e0bf347bb355",\n    "llava/mm_utils.py": "de0a0028b763fc346272c46ab1a5a67156d64615c8d931dbe7525e7d48210257",\n    "llava/model/__init__.py": "17f5ce845351631fa1345701ea214ef053633d74e693e360c29b8010b88b8ee2",\n    "llava/model/builder.py": "cff9dc7a451cb41ea7c85ed33b67528821a39253e2281760dd5d73db50dc7463",\n    "llava/model/language_model/llava_qwen.py": "4e0182c0c6ad0b6873450477a31568204661e1bb007cac0f55a767805f6834f1",\n    "llava/model/llava_arch.py": "910a1dd3e8dce5fed149e5f6d52045f13bcc87fe1519d2527178a618480eeda3",\n    "llava/model/multimodal_encoder/base_encoder.py": "914f5ffeddd5251abf6cb7b73bbd2dd7f506af64dd6924b454ed2bab1a5d2e3d",\n    "llava/model/multimodal_encoder/builder.py": "060fc4b3843f4a31ddbbbfbc5508a0d025517454f17660accb7df5c7abdfc495",\n    "llava/model/multimodal_encoder/siglip_encoder.py": "400009265fe69bc207289c3cac88c15d29c156176f6bee793b34137ea0536641",\n    "llava/model/multimodal_projector/builder.py": "742da4d2c11daa8d7c5d3d0e72f33237303efa4a20612fd8e693c82b53fc2c3b",\n    "llava/model/multimodal_resampler/builder.py": "a64bc968dc415c18b6ee693312e6db89857ff007a077dbcc8289cbbd4a78a6c7",\n    "llava/model/multimodal_resampler/mdp.py": "81b08467d7b93e70442007f0d3a9315d0d006f28ab66f75512b7164244874590",\n    "llava/model/utils.py": "e6eaf4aa4ba9983842c5b26de9641cd2462a4fa760386932286c680dad296b38",\n    "llava/train/llama_flash_attn_monkey_patch.py": "4262998fd2e4e0e26ca58e90eda710c468f1f7f38129dbe7d16c01349c527fc4",\n    "llava/train/llama_xformers_attn_monkey_patch.py": "dac740469091f03780b888d0fe82f590e5afaa14e709f0916bad2e647f5b03d5",\n    "llava/train/llava_trainer.py": "10e1290d4c0ecbb8251833ee635962409ec61a60c2892f3d26a2a5439fc3e339",\n    "llava/train/train.py": "51b4aad61d797b56566e081210ede15228f6bd7fcef1b8685e9fcf21c20a00b0",\n    "llava/train/train_mem.py": "a608c9a3e24504d16c252febf52b4b6bd0f9ed393148abb0570a67d8f3837e5d",\n    "llava/transformers.py": "86b51404d94a24203758f60ef550ab4f31f7c812d88e01eabda3514e142934af",\n    "llava/utils.py": "0442ed85500257af7144bbff2e5d69b6565b95ee12fbda95bce32a2dbd317630",\n    "pyproject.toml": "6edacf8cb3a72673e3ac0e821fe9c721a03c4b7e563f48561c4b7ec3dfb96159",\n    "README.md": "7c6ba170638275124a56e4a6349e9c7e50486250670cb51b8f10fb32efc22be0",\n    "LICENSE": "1eb85fc97224598dad1852b5d6483bbcf0aa8608790dcc657a5a2a761ae9c8c6"\n  },\n  "question": "What modifications can be observed between these two pictures?  <image><image>Difference:",\n  "generation": {\n    "do_sample": false,\n    "num_beams": 1,\n    "max_new_tokens": 256,\n    "temperature": 1.0,\n    "top_p": 1.0,\n    "top_k": 50,\n    "no_repeat_ngram_size": 0,\n    "min_new_tokens": 0,\n    "pad_token_id": 151645,\n    "eos_token_id": 151645,\n    "bos_token_id": null,\n    "use_cache": true,\n    "return_dict_in_generate": false,\n    "output_scores": false,\n    "repetition_penalty": 1.05\n  },\n  "lora": {\n    "r": 16,\n    "lora_alpha": 32,\n    "lora_dropout": 0.0,\n    "bias": "none",\n    "task_type": "CAUSAL_LM",\n    "target_suffixes": [\n      ".q_proj",\n      ".v_proj"\n    ]\n  },\n  "seed": 20261031\n}\n', 'inference_tokens.py': "def token_layout(tokenizer, row, answer=True):\n    start = tokenizer.convert_tokens_to_ids('<|im_start|>')\n    end = tokenizer.convert_tokens_to_ids('<|im_end|>')\n    if start == end or start == tokenizer.unk_token_id or end == tokenizer.unk_token_id:\n        raise ValueError('Wrong Qwen tokenizer')\n    enc = lambda s: tokenizer.encode(s, add_special_tokens=False)\n    ids = [start] + enc('system\\nYou are a helpful assistant.') + [end] + enc('\\n')\n    labels = [-100] * len(ids)\n    for turn in row['conversations']:\n        role = {'human': 'user', 'gpt': 'assistant'}[turn['from']]\n        if not answer and role != 'user':\n            continue\n        header = [start] + enc(role + '\\n')\n        body = []\n        for i, chunk in enumerate(turn['value'].split('<image>')):\n            if i:\n                body.append(-200)\n            body.extend(enc(chunk))\n        segment = header + body + [end] + enc('\\n')\n        ids += segment\n        labels += [-100] * len(segment) if role == 'user' else [-100] * len(header) + body + [end] + enc('\\n')\n    if not answer:\n        tail = [start] + enc('assistant\\n')\n        ids += tail\n        labels += [-100] * len(tail)\n    if ids.count(-200) != 2 or any((y != -100 for x, y in zip(ids, labels) if x == -200)):\n        raise ValueError('Image token masking')\n    if answer and (not any((y != -100 for y in labels))):\n        raise ValueError('No answer labels')\n    if len(ids) + 1458 > 4096:\n        raise ValueError('Sequence exceeds 4096; no silent truncation')\n    return (ids, labels)\n", 'model_runner.py': '"""Persistent inference using the author\'s OmniDiff-Real preprocessing."""\n\nfrom __future__ import annotations\n\nfrom copy import deepcopy\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib\nfrom importlib import metadata\nimport json\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport threading\nfrom time import perf_counter\nimport uuid\n\nfrom PIL import Image\n\n\ndef _sha256(path: Path) -> str:\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef _canonical_hash(value: dict) -> str:\n    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False).encode()).hexdigest()\n\n\ndef _validate_protocol(protocol: dict) -> dict:\n    result = deepcopy(protocol)\n    required = {\n        \'prompt\', \'conv_mode\', \'torch_dtype\', \'flash_attention\', \'temperature\',\n        \'top_p\', \'num_beams\', \'max_new_tokens\', \'use_cache\',\n    }\n    if set(result) != required:\n        raise ValueError(f\'Protocol keys must be exactly: {sorted(required)}\')\n    if not isinstance(result[\'prompt\'], str) or result[\'prompt\'].count(\'<image>\') != 2:\n        raise ValueError(\'Prompt must contain exactly two <image> tokens\')\n    if result[\'torch_dtype\'] not in {\'float16\', \'bfloat16\', \'float32\'}:\n        raise ValueError(\'Unsupported torch dtype\')\n    for name in (\'max_new_tokens\', \'num_beams\'):\n        if type(result[name]) is not int or result[name] < 1:\n            raise ValueError(f\'{name} must be a positive integer\')\n    if not isinstance(result[\'temperature\'], (int, float)) or result[\'temperature\'] < 0:\n        raise ValueError(\'temperature must be nonnegative\')\n    if result[\'top_p\'] is not None and not 0 < result[\'top_p\'] <= 1:\n        raise ValueError(\'top_p must be None or in (0, 1]\')\n    if type(result[\'use_cache\']) is not bool or type(result[\'flash_attention\']) is not bool:\n        raise ValueError(\'use_cache and flash_attention must be booleans\')\n    return result\n\n\nclass M3DiffRunner:\n    """One loaded checkpoint, reused for each sequential compare request."""\n\n    def __init__(self, model_path: str, protocol: dict):\n        self._protocol = _validate_protocol(protocol)\n        self.model_path = Path(model_path).expanduser().absolute()\n        if not (self.model_path / \'config.json\').is_file():\n            raise FileNotFoundError(f\'Checkpoint config missing: {self.model_path}\')\n        self._torch = importlib.import_module(\'torch\')\n        if not self._torch.cuda.is_available():\n            raise RuntimeError(\'M3Diff requires the prepared Colab CUDA runtime\')\n        self._evaluation = importlib.import_module(\'llava.eval.model_vqa_qwen_OmniDiff_Real\')\n        if self._protocol[\'conv_mode\'] not in self._evaluation.conv_templates:\n            raise ValueError(\'Unknown conversation template\')\n        self._dtype = getattr(self._torch, self._protocol[\'torch_dtype\'])\n        self._lock = threading.Lock()\n        self.model_instance_id = uuid.uuid4().hex\n        self.successful_requests = 0\n        self.model_load_count = 0\n\n        self._evaluation.disable_torch_init()\n        started = perf_counter()\n        self.tokenizer, self.model, self.image_processor, _ = self._evaluation.load_pretrained_model(\n            str(self.model_path), None,\n            self._evaluation.get_model_name_from_path(str(self.model_path)),\n            use_flash_attn=self._protocol[\'flash_attention\'], torch_dtype=self._dtype,\n        )\n        self.model.eval()\n        self._torch.cuda.synchronize()\n        self.load_seconds = perf_counter() - started\n        self.model_load_count += 1\n        self.runtime_manifest = self._make_manifest()\n        self.protocol_id = _canonical_hash({\n            \'protocol\': self._protocol,\n            \'source_sha256\': self.runtime_manifest[\'source_sha256\'],\n            \'checkpoint_config_sha256\': self.runtime_manifest[\'checkpoint_config_sha256\'],\n            \'package_versions\': self.runtime_manifest[\'package_versions\'],\n        })\n\n    @property\n    def protocol(self) -> dict:\n        return deepcopy(self._protocol)\n\n    def _make_manifest(self) -> dict:\n        source_root = Path(self._evaluation.__file__).resolve().parents[2]\n        names = (\n            \'llava/eval/model_vqa_qwen_OmniDiff_Real.py\',\n            \'llava/model/builder.py\', \'llava/model/llava_arch.py\',\n            \'llava/model/multimodal_resampler/mdp.py\',\n            \'llava/model/multimodal_resampler/builder.py\',\n        )\n        hashes = {name: _sha256(source_root / name) for name in names if (source_root / name).is_file()}\n        try:\n            revision = subprocess.check_output(\n                [\'git\', \'-C\', str(source_root), \'rev-parse\', \'HEAD\'],\n                text=True, stderr=subprocess.DEVNULL, timeout=10,\n            ).strip()\n        except (OSError, subprocess.SubprocessError):\n            revision = None\n        versions = {}\n        for name in (\'torch\', \'torchvision\', \'transformers\', \'tokenizers\', \'huggingface-hub\', \'numpy\', \'pillow\'):\n            try:\n                versions[name] = metadata.version(name)\n            except metadata.PackageNotFoundError:\n                versions[name] = None\n        return {\n            \'created_at_utc\': datetime.now(timezone.utc).isoformat(),\n            \'python\': platform.python_version(), \'source_root\': str(source_root),\n            \'source_revision\': revision, \'source_sha256\': hashes,\n            \'package_versions\': versions, \'gpu\': self._torch.cuda.get_device_name(0),\n            \'checkpoint_path\': str(self.model_path),\n            \'checkpoint_config_sha256\': _sha256(self.model_path / \'config.json\'),\n            \'checkpoint_shards\': {\n                path.name: path.stat().st_size for path in sorted(self.model_path.glob(\'*.safetensors\'))\n            },\n            \'checkpoint_identity_note\': \'Config hash and shard sizes only; weights were not fully hashed.\',\n        }\n\n    def compare(self, before: Image.Image, after: Image.Image) -> dict:\n        if not isinstance(before, Image.Image) or not isinstance(after, Image.Image):\n            raise TypeError(\'compare expects two PIL images, in before/after order\')\n        with self._lock:\n            self._torch.cuda.synchronize()\n            started = perf_counter()\n            evaluation = self._evaluation\n            settings = self._protocol\n            conv = evaluation.conv_templates[settings[\'conv_mode\']].copy()\n            input_ids = evaluation.preprocess_qwen(\n                [{\'from\': \'human\', \'value\': settings[\'prompt\']}, {\'from\': \'gpt\', \'value\': None}],\n                self.tokenizer, has_image=True,\n            ).cuda()\n            image_tensors = [\n                evaluation.process_images([image.convert(\'RGB\')], self.image_processor, self.model.config).to(self._dtype)\n                for image in (before, after)\n            ]\n            image_tensors = [self._torch.cat(image_tensors, dim=0)]\n            stop_str = conv.sep if conv.sep_style != evaluation.SeparatorStyle.TWO else conv.sep2\n            stopping = evaluation.KeywordsStoppingCriteria([stop_str], self.tokenizer, input_ids)\n            with self._torch.inference_mode():\n                output_ids = self.model.generate(\n                    inputs=input_ids, images=image_tensors,\n                    do_sample=settings[\'temperature\'] > 0, temperature=settings[\'temperature\'],\n                    top_p=settings[\'top_p\'], num_beams=settings[\'num_beams\'],\n                    max_new_tokens=settings[\'max_new_tokens\'], use_cache=settings[\'use_cache\'],\n                    stopping_criteria=[stopping],\n                )\n            caption = self.tokenizer.batch_decode(output_ids, skip_special_tokens=True)[0].strip()\n            if stop_str and caption.endswith(stop_str):\n                caption = caption[:-len(stop_str)].strip()\n            if not caption:\n                raise RuntimeError(\'Model returned an empty caption\')\n            self._torch.cuda.synchronize()\n            self.successful_requests += 1\n            return {\n                \'caption\': caption, \'elapsed_seconds\': round(perf_counter() - started, 4),\n                \'protocol_id\': self.protocol_id, \'model_instance_id\': self.model_instance_id,\n                \'model_load_count\': self.model_load_count,\n                \'successful_requests\': self.successful_requests,\n            }\n', 'model_worker.py': '"""Run one persistent M3Diff model in the isolated Python interpreter."""\n\nimport argparse\nfrom contextlib import redirect_stdout\nimport json\nfrom pathlib import Path\nimport sys\nimport traceback\n\nPREFIX = \'M3DIFF_RPC:\'\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'--model-path\', required=True)\n    parser.add_argument(\'--protocol\', required=True)\n    args = parser.parse_args()\n    output = sys.stdout\n\n    def reply(payload):\n        output.write(PREFIX + json.dumps(payload, ensure_ascii=True) + \'\\n\')\n        output.flush()\n\n    def info():\n        return {\n            \'model_path\': str(runner.model_path), \'protocol\': runner.protocol,\n            \'model_instance_id\': runner.model_instance_id,\n            \'model_load_count\': runner.model_load_count,\n            \'successful_requests\': runner.successful_requests,\n            \'load_seconds\': runner.load_seconds, \'protocol_id\': runner.protocol_id,\n            \'runtime_manifest\': runner.runtime_manifest,\n            \'worker_python\': sys.executable, \'worker_version\': sys.version,\n        }\n\n    try:\n        with redirect_stdout(sys.stderr):\n            from PIL import Image\n            protocol = json.loads(Path(args.protocol).read_text(encoding=\'utf-8\'))\n            if protocol.get(\'model_variant\') == \'B_SCST_128\':\n                from scst_demo_runner import SCSTRunner\n                runner = SCSTRunner(args.model_path, protocol)\n            elif \'model_variant\' not in protocol:\n                from model_runner import M3DiffRunner\n                runner = M3DiffRunner(args.model_path, protocol)\n            else:\n                raise ValueError(\'Unknown model variant; refusing to fall back to G\')\n        reply({\'ok\': True, \'result\': info()})\n    except Exception as exc:\n        traceback.print_exc(file=sys.stderr)\n        reply({\'ok\': False, \'error\': f\'{type(exc).__name__}: {exc}\'})\n        return 1\n\n    for line in sys.stdin:\n        try:\n            request = json.loads(line)\n            action = request[\'action\']\n            if action == \'shutdown\':\n                reply({\'ok\': True, \'result\': {\'closed\': True}})\n                break\n            with redirect_stdout(sys.stderr):\n                if action == \'info\':\n                    result = info()\n                elif action == \'compare\':\n                    with Image.open(request[\'before\']) as before, Image.open(request[\'after\']) as after:\n                        result = runner.compare(before.convert(\'RGB\'), after.convert(\'RGB\'))\n                else:\n                    raise ValueError(f\'Unknown action: {action}\')\n            reply({\'ok\': True, \'result\': result})\n        except Exception as exc:\n            traceback.print_exc(file=sys.stderr)\n            reply({\'ok\': False, \'error\': f\'{type(exc).__name__}: {exc}\'})\n    return 0\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main())\n', 'worker_client.py': '"""Notebook bridge to a model process; imports no Torch or Transformers."""\n\nimport atexit\nimport json\nfrom pathlib import Path\nimport queue\nimport subprocess\nfrom tempfile import TemporaryDirectory\nimport threading\n\nPREFIX = \'M3DIFF_RPC:\'\n\n\nclass WorkerClient:\n    def __init__(self, python, worker, model_path, protocol, work, *, source=None, timeout=1800):\n        self.timeout = timeout\n        self.work = Path(work)\n        self.work.mkdir(parents=True, exist_ok=True)\n        self.log_path = self.work / \'model_worker.log\'\n        self._messages = queue.Queue()\n        self._lock = threading.Lock()\n        self._log = self.log_path.open(\'a\', encoding=\'utf-8\')\n        self._log.write(\'\\n--- Starting isolated model worker ---\\n\')\n        self._log.flush()\n        try:\n            self.process = subprocess.Popen(\n                [str(python), \'-u\', \'-B\', str(worker), \'--model-path\', str(model_path), \'--protocol\', str(protocol)],\n                cwd=str(source or self.work), stdin=subprocess.PIPE, stdout=subprocess.PIPE,\n                stderr=self._log, text=True, encoding=\'utf-8\', bufsize=1,\n            )\n        except Exception:\n            self._log.close()\n            raise\n        threading.Thread(target=self._read_responses, daemon=True).start()\n        try:\n            self.initial_info = self._receive()\n        except BaseException:\n            self.close()\n            raise\n        atexit.register(self.close)\n\n    def _read_responses(self):\n        try:\n            for line in self.process.stdout:\n                if line.startswith(PREFIX):\n                    self._messages.put(json.loads(line[len(PREFIX):]))\n        except Exception as exc:\n            self._messages.put({\'ok\': False, \'error\': f\'Invalid worker response: {exc}\'})\n        finally:\n            self._messages.put(None)\n\n    def _receive(self):\n        try:\n            message = self._messages.get(timeout=self.timeout)\n        except queue.Empty:\n            self.close()  # A late response must never be consumed as the next request.\n            raise RuntimeError(f\'Model worker timed out; stopped it. See {self.log_path}\') from None\n        if message is None:\n            raise RuntimeError(f\'Model worker stopped. See {self.log_path}\')\n        if not message[\'ok\']:\n            raise RuntimeError(f"{message[\'error\']}\\nSee {self.log_path}")\n        return message[\'result\']\n\n    def _request(self, payload):\n        with self._lock:\n            if self.process.poll() is not None:\n                raise RuntimeError(f\'Model worker is no longer running. See {self.log_path}\')\n            try:\n                self.process.stdin.write(json.dumps(payload) + \'\\n\')\n                self.process.stdin.flush()\n            except BaseException:\n                self.close()\n                raise\n            try:\n                return self._receive()\n            except (KeyboardInterrupt, SystemExit):\n                # A late reply must never be read as the next image pair\'s result.\n                self.close()\n                raise\n\n    def info(self):\n        return self._request({\'action\': \'info\'})\n\n    def compare(self, before, after):\n        with TemporaryDirectory(prefix=\'m3diff_request_\', dir=self.work) as directory:\n            before_path = Path(directory) / \'before.png\'\n            after_path = Path(directory) / \'after.png\'\n            before.convert(\'RGB\').save(before_path)\n            after.convert(\'RGB\').save(after_path)\n            return self._request({\'action\': \'compare\', \'before\': str(before_path), \'after\': str(after_path)})\n\n    def close(self):\n        process = self.process\n        if process.poll() is None:\n            try:\n                process.stdin.write(\'{"action":"shutdown"}\\n\')\n                process.stdin.flush()\n                process.wait(timeout=5)\n            except (OSError, subprocess.TimeoutExpired):\n                process.terminate()\n                try:\n                    process.wait(timeout=5)\n                except subprocess.TimeoutExpired:\n                    process.kill()\n                    process.wait(timeout=5)\n        if process.stdin and not process.stdin.closed:\n            process.stdin.close()\n        if process.stdout and not process.stdout.closed:\n            process.stdout.close()\n        if not self._log.closed:\n            self._log.close()\n'}
SOURCE_ZIP_BASE64 = '...'
for name, content in SUPPORT_FILES.items():
    (WORK/name).write_text(content,encoding='utf-8')
release=json.loads(SUPPORT_FILES['scst_release.json'])
archive_bytes=base64.b64decode(SOURCE_ZIP_BASE64)
if hashlib.sha256(archive_bytes).hexdigest()!=release['source_sha256']:
    raise ValueError('Source snapshot hỏng; chưa cài môi trường.')
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as z:
    for item in z.infolist():
        relative=PurePosixPath(item.filename)
        if relative.is_absolute() or '..' in relative.parts or '\\' in item.filename or ':' in item.filename:
            raise ValueError('Unsafe source path')
        if not item.is_dir():
            target=SOURCE/item.filename;target.parent.mkdir(parents=True,exist_ok=True)
            target.write_bytes(z.read(item))
for name,digest in release['source_files'].items():
    if hashlib.sha256((SOURCE/name).read_bytes()).hexdigest()!=digest:
        raise ValueError('Sai source: '+name)
if importlib.util.find_spec('uv') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
uv=[sys.executable,'-m','uv']
if not VENV_PY.exists():
    subprocess.run(uv+['python','install','3.10'],check=True)
    subprocess.run(uv+['venv',str(VENV),'--python','3.10'],check=True)
version=subprocess.check_output([str(VENV_PY),'-c','import sys; print("%d.%d" % sys.version_info[:2])'],text=True).strip()
if version!='3.10':raise RuntimeError('Demo cần venv Python3.10; không cài đè môi trường khác.')
expected={'torch':'2.2.2','torchvision':'0.17.2','transformers':'4.40.0','tokenizers':'0.19.1',
    'huggingface-hub':'0.36.2','numpy':'1.26.4','pillow':'11.3.0','accelerate':'0.29.1',
    'einops':'0.6.1','einops-exts':'0.0.4','sentencepiece':'0.1.99','bitsandbytes':'0.41.0',
    'requests':'2.32.3','peft':'0.10.0','timm':'0.9.12','shortuuid':'1.0.13',
    'scipy':'1.11.4','setuptools':'69.5.1'}
probe="""import json,sys
from importlib import metadata
out={}
for name in json.loads(sys.argv[1]):
    try: out[name]=metadata.version(name).split('+')[0]
    except metadata.PackageNotFoundError: out[name]=None
print(json.dumps(out))
"""
installed=json.loads(subprocess.check_output([str(VENV_PY),'-c',probe,json.dumps(list(expected))],text=True))
pip=uv+['pip','install','--python',str(VENV_PY)]
if any(installed[n]!=expected[n] for n in ['torch','torchvision']):
    subprocess.run(pip+['torch==2.2.2','torchvision==0.17.2','--index-url','https://download.pytorch.org/whl/cu121'],check=True)
deps=[n+'=='+v for n,v in expected.items() if n not in ['torch','torchvision'] and installed[n]!=v]
if deps:subprocess.run(pip+deps,check=True)
# Explicit source path, inherited by model worker; no change to the project's clone.
os.environ['PYTHONPATH']=str(SOURCE)+os.pathsep+str(WORK)
preflight="""import torch, scipy, bitsandbytes
from peft import LoraConfig, get_peft_model, get_peft_model_state_dict, set_peft_model_state_dict
from transformers import AutoTokenizer, GenerationConfig
from llava.model.language_model.llava_qwen import LlavaQwenForCausalLM
from llava.mm_utils import process_images
import scst_artifacts, scst_demo_runner
print('IMPORT OK — chưa nạp model.')
"""
subprocess.run([str(VENV_PY),'-c',preflight],cwd=SOURCE,check=True)
print('Sẵn sàng. Chạy ô2 để dùng model và checkpoint trên Drive.')


In [ ]:
#@title 2. Drive, model gốc và đúng adapter B128 — dùng lại dữ liệu đã có
if 'SUPPORT_FILES' not in globals():raise RuntimeError('Chạy ô1 trước.')
if 'runner' in globals() and runner.process.poll() is None:
    if runner.info()['protocol'].get('model_variant') != 'B_SCST_128':
        raise RuntimeError('Model G cũ còn trong GPU. Chạy runner.close() trước khi dùng demo B.')
from google.colab import drive
drive.mount('/content/drive')
STORAGE=Path('/content/drive/MyDrive/M3Diff_Storage')
gpu_check="""import torch
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), 'Chọn GPU A100 hỗ trợ BF16.'
print('GPU:',torch.cuda.get_device_name(0))
"""
subprocess.run([str(VENV_PY),'-c',gpu_check],check=True)
subprocess.run([str(VENV_PY),str(WORK/'scst_artifacts.py'),'prepare',
    '--storage',str(STORAGE),'--release',str(WORK/'scst_release.json'),
    '--protocol',str(WORK/'protocol.json')],cwd=SOURCE,check=True)
protocol=json.loads((WORK/'protocol.json').read_text(encoding='utf-8'))
print('CHECKPOINT B128 ĐẠT. Chạy ô3; không huấn luyện thêm.')


In [ ]:
#@title 3. Nạp B128 một lần — chọn ảnh trước rồi ảnh sau để kiểm tra
from pathlib import Path
import json, sys, subprocess
if not all(name in globals() for name in ('WORK','SOURCE','MODEL_PATH','VENV_PY')):
    raise RuntimeError('Chạy Ô 1 và Ô 2 trước khi nạp mô hình.')
WORK.mkdir(parents=True, exist_ok=True)

# Mã hỗ trợ được nhúng để notebook dùng độc lập.
protocol=json.loads((WORK/'protocol.json').read_text(encoding='utf-8'))

# Ô 6 — TẢI ẢNH TRƯỚC/SAU (THỨ TỰ QUAN TRỌNG)
from google.colab import files
from PIL import Image
from io import BytesIO
from tempfile import TemporaryDirectory
from pathlib import Path
import hashlib, os

def upload_one_image(label):
    print(label)
    previous_cwd = Path.cwd()
    with TemporaryDirectory(prefix='m3diff_upload_', dir=str(WORK)) as temporary:
        try:
            os.chdir(temporary)
            uploaded = files.upload()
        finally:
            os.chdir(previous_cwd)
        if len(uploaded) != 1:
            raise ValueError('Mỗi lượt chọn đúng một ảnh.')
        name, data = next(iter(uploaded.items()))
        with Image.open(BytesIO(data)) as opened:
            image = opened.convert('RGB').copy()
    return image, {'name': Path(name).name, 'sha256': hashlib.sha256(data).hexdigest(), 'size': list(image.size)}


def show_pair(before, after):
    from IPython.display import display
    for label, image in [('ẢNH TRƯỚC', before), ('ẢNH SAU', after)]:
        print(label)
        preview = image.copy()
        preview.thumbnail((640, 360))
        display(preview)

def save_report():
    # Lưu sau mỗi lượt; Ô 5 không phụ thuộc model còn sống.
    if 'smoke_report' not in globals():
        return
    content = json.dumps(smoke_report, ensure_ascii=False, indent=2)
    (WORK / 'report_latest.json').write_text(content, encoding='utf-8')
    session_id = smoke_report['requests'][0]['model_instance_id']
    session_dir = WORK / 'sessions' / session_id
    session_dir.mkdir(parents=True, exist_ok=True)
    (session_dir / 'report.json').write_text(content, encoding='utf-8')

def prepare_demo():
    global runner, smoke_report, protocol
    import json
    if 'runner' in globals():
        existing = runner
        if existing.process.poll() is None:
            info = existing.info()
            if info['protocol'] != protocol or Path(info['model_path']) != MODEL_PATH.absolute():
                raise RuntimeError('Model đã có cấu hình khác. Dùng notebook trong phiên riêng để tránh đổi cấu hình.')
            if ('smoke_report' in globals()
                    and smoke_report['requests'][0]['model_instance_id'] == info['model_instance_id']):
                print('MODEL ĐÃ SẴN SÀNG — dùng lại, giữ nguyên lịch sử.')
                print('Chạy Ô 4 cho cặp mới, Ô 5 để tải báo cáo.')
                return
        # Worker dừng hoặc lần kiểm tra trước chưa hoàn tất: đóng trước khi nạp lại.
        existing.close()
        del runner
    save_report()
    if 'smoke_report' in globals():
        del smoke_report  # Báo cáo cũ đã lưu; worker mới phải được kiểm tra riêng.
    print('Lần đầu: chọn một cặp ảnh để kiểm tra cách chạy. Các cặp sau dùng Ô 4.')
    before, before_info = upload_one_image('1/2 — Chọn ảnh TRƯỚC')
    after, after_info = upload_one_image('2/2 — Chọn ảnh SAU')
    print('Đang chuẩn bị B128; chỉ nạp một model, không chạy lại G.')

    # Nạp một lần trong Python 3.10
    import importlib.util, json
    protocol = json.loads((WORK / 'protocol.json').read_text())
    if 'runner' not in globals():
        spec = importlib.util.spec_from_file_location('m3diff_worker_client', WORK / 'worker_client.py')
        client_module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(client_module)
        print('Đang nạp model bằng Python 3.10. Log:', WORK / 'model_worker.log', flush=True)
        runner = client_module.WorkerClient(VENV_PY, WORK / 'model_worker.py', MODEL_PATH,
                                           WORK / 'protocol.json', WORK, source=SOURCE)
    else:
        print('Dùng lại tiến trình giữ model đã có.')
    model_info = runner.info()
    if model_info['protocol'] != protocol or Path(model_info['model_path']) != MODEL_PATH.absolute():
        raise RuntimeError('Model đã có cấu hình khác; cần runner.close() rồi xóa biến runner trước khi nạp lại.')
    print('MODEL READY:', model_info['model_instance_id'])
    print('Python model:', model_info['worker_python'], model_info['worker_version'])
    print('Số lần nạp:', model_info['model_load_count'], '| Thời gian nạp:', round(model_info['load_seconds'], 2), 'giây')
    print('Protocol ID:', model_info['protocol_id'])

    # Ô 9 — HAI LƯỢT QUA CÙNG TIẾN TRÌNH PYTHON 3.10
    from datetime import datetime, timezone
    import json

    initial = runner.info()
    first = runner.compare(before, after)
    second = runner.compare(before, after)
    current = runner.info()
    reuse_ok = (initial['model_instance_id'] == first['model_instance_id'] == second['model_instance_id'] == current['model_instance_id']
                and current['model_load_count'] == 1
                and current['successful_requests'] == initial['successful_requests'] + 2)
    adapter_ok = current['runtime_manifest'].get('model_variant') == 'B_SCST_128'
    smoke_report = {
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'scope': 'one_pair_integration_check_not_benchmark',
        'inputs': {'before': before_info, 'after': after_info},
        'protocol': current['protocol'], 'protocol_id': current['protocol_id'],
        'runtime_manifest': current['runtime_manifest'], 'load_seconds': current['load_seconds'],
        'worker_python': current['worker_python'], 'worker_version': current['worker_version'],
        'notebook_python': sys.version,
        'model_variant': 'B_SCST_128', 'adapter_verified': adapter_ok,
        'same_pair_caption_repeatable': first['caption'] == second['caption'],
        'model_reuse_check_passed': reuse_ok, 'requests': [first, second], 'additional_pairs': [],
    }
    smoke_report['status'] = (
        'integration_check_passed' if reuse_ok and adapter_ok and first['caption'] == second['caption']
        else 'needs_review_or_missing_reference'
    )
    for index, result in enumerate([first, second], 1):
        print(f"LẦN {index} ({result['elapsed_seconds']} giây): {result['caption']}")
    print('Giữ nguyên model:', reuse_ok)
    print('Đúng adapter B128:', adapter_ok)
    print('Trạng thái:', smoke_report['status'])
    if not reuse_ok:
        raise RuntimeError('Kiểm tra dùng lại model thất bại. Chạy Ô 5 để tải báo cáo.')
    if not adapter_ok or first['caption'] != second['caption']:
        print('Cần xem lại adapter hoặc độ lặp lại của caption; gửi ZIP ô5.')

    save_report()
    show_pair(before, after)
    print('MÔ TẢ:', first['caption'])
    print('KIỂM TRA ĐẠT' if smoke_report['status'] == 'integration_check_passed' else 'CẦN XEM LẠI — tải ZIP ở Ô 5')
    print('Từ giờ chạy Ô 4 để so sánh cặp mới; Ô 5 để tải báo cáo.')

setup_error = None
try:
    prepare_demo()
    (WORK / 'setup_error.json').unlink(missing_ok=True)
except Exception as exc:
    setup_error = {'stage':'cell_3','error':str(exc)}
    (WORK / 'setup_error.json').write_text(json.dumps(setup_error, ensure_ascii=False, indent=2), encoding='utf-8')
    if 'smoke_report' in globals():
        save_report()
    log = WORK / 'model_worker.log'
    if log.is_file():
        print(log.read_text(encoding='utf-8', errors='replace')[-4000:])
    print('Chạy Ô 5 để tải thông tin lỗi. Nếu đã bấm dừng, chạy lại Ô 3.')
    raise


In [ ]:
#@title 4. So sánh hai ảnh — chạy lại ô này cho mỗi cặp mới
if 'runner' not in globals() or 'smoke_report' not in globals():
    raise RuntimeError('Chạy Ô 3 để nạp và kiểm tra mô hình trước.')
if runner.process.poll() is not None:
    raise RuntimeError('Model đã dừng. Chạy lại Ô 3 để nạp lại rồi quay về ô này.')

# So sánh bằng model đã giữ trong bộ nhớ
active = runner.info()
if (not smoke_report.get('requests')
        or active['model_instance_id'] != smoke_report['requests'][0]['model_instance_id']):
    raise RuntimeError('Model hiện tại chưa có kết quả kiểm tra tương ứng. Chạy lại Ô 3 trước.')
new_before, new_before_info = upload_one_image('1/2 — Cặp mới: ảnh TRƯỚC')
new_after, new_after_info = upload_one_image('2/2 — Cặp mới: ảnh SAU')
existing = runner.info()
new_result = runner.compare(new_before, new_after)
current = runner.info()
if current['model_instance_id'] != existing['model_instance_id'] or current['model_load_count'] != 1:
    raise RuntimeError('Model đã đổi ngoài dự kiến.')
smoke_report['additional_pairs'].append({'before':new_before_info, 'after':new_after_info, 'result':new_result})
print('KẾT QUẢ:', new_result['caption'])
print('Thời gian:', new_result['elapsed_seconds'], 'giây | Số lần nạp model:', current['model_load_count'])

save_report()
show_pair(new_before, new_after)
print('Đã lưu kết quả. Chạy lại Ô 4 cho cặp mới hoặc Ô 5 để tải báo cáo.')


In [ ]:
#@title 5. Tải ZIP báo cáo — có thể chạy sau khi model đã dừng
from datetime import datetime, timezone
from pathlib import Path
import json, zipfile
from google.colab import files

WORK = Path(globals().get('WORK', '/content/m3diff_demo'))
if 'smoke_report' in globals():
    report = smoke_report
elif (WORK / 'report_latest.json').is_file():
    report = json.loads((WORK / 'report_latest.json').read_text(encoding='utf-8'))
else:
    report = {'status':'setup_incomplete','scope':'diagnostic_only_not_benchmark'}
if globals().get('setup_error'):
    report = {'status':'setup_failed','scope':'diagnostic_only_not_benchmark',
              'latest_setup_error':setup_error,'previous_completed_report':report}
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
report_dir = WORK / 'reports' / stamp
report_dir.mkdir(parents=True, exist_ok=False)
(report_dir / 'report.json').write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')
saved_protocol = report.get('protocol')
if saved_protocol is None and (WORK / 'protocol.json').is_file():
    saved_protocol = json.loads((WORK / 'protocol.json').read_text(encoding='utf-8'))
if saved_protocol is not None:
    (report_dir / 'protocol.json').write_text(json.dumps(saved_protocol, ensure_ascii=False, indent=2), encoding='utf-8')
archive = report_dir.with_suffix('.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as zipped:
    for path in sorted(report_dir.iterdir()):
        zipped.write(path, arcname=path.name)
    for name in ('model_worker.log', 'setup_error.json'):
        if (WORK / name).is_file():
            zipped.write(WORK / name, arcname=name)
print('Báo cáo:', archive)
print('ZIP gồm caption, lịch sử, cấu hình và log; không chứa ảnh gốc hoặc trọng số.')
print('Trạng thái:', report['status'], '— kiểm tra demo, chưa phải kết quả benchmark.')
files.download(str(archive))


## Kết nối với web — chạy ô 6 rồi ô 7
Hai ô này dùng model của ô 3, không nạp model mới.


In [ ]:
#@title 6. Chuẩn bị kết nối web — bấm chạy, không nạp lại model
# Ô 6 — dùng runner đã nạp ở ô 3 của notebook này
import sys, subprocess, importlib, importlib.metadata
from pathlib import Path
if 'runner' not in globals() or 'WORK' not in globals():
    raise RuntimeError('Chưa có model. Chạy ô 1 → 2 → 3 của notebook này, rồi quay lại ô 6.')
if hasattr(runner, 'process') and runner.process.poll() is not None:
    raise RuntimeError('Worker đã dừng. Khởi động lại runner bằng ô 3 của notebook gốc trước.')
web_info = runner.info()  # chỉ đọc thông tin, KHÔNG nạp model
try:
    gradio_version = importlib.metadata.version('gradio')
except importlib.metadata.PackageNotFoundError:
    gradio_version = ''
if not gradio_version or gradio_version.split('.')[0] not in ('5', '6'):
    if 'gradio' in sys.modules:
        raise RuntimeError('Kernel đã import Gradio quá cũ. Không cài đè trong phiên đang chạy; gửi lỗi này để kiểm tra.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'gradio==5.49.1'])
    importlib.invalidate_caches()
# Chỉ cài giao diện trong kernel; môi trường Python 3.10 của model được giữ nguyên.
WEB_SOURCES = {'images.py': '"""Image validation and crops in coordinates of the EXIF-normalized original."""\n\nimport base64\nfrom io import BytesIO\nimport math\nimport warnings\n\nfrom PIL import Image, ImageOps, UnidentifiedImageError\n\nMAX_BYTES = 20 * 1024 * 1024\nMAX_PIXELS = 25_000_000\n\n\ndef normalize_image(data: bytes) -> Image.Image:\n    if not data or len(data) > MAX_BYTES:\n        raise ValueError(\'Mỗi ảnh cần nhỏ hơn 20 MB và có nội dung.\')\n    try:\n        with warnings.catch_warnings():\n            warnings.simplefilter(\'error\', Image.DecompressionBombWarning)\n            with Image.open(BytesIO(data)) as image:\n                if image.format not in (\'JPEG\', \'PNG\'):\n                    raise ValueError(\'Chọn ảnh JPG hoặc PNG.\')\n                if image.width * image.height > MAX_PIXELS:\n                    raise ValueError(\'Ảnh vượt giới hạn 25 triệu pixel.\')\n                return ImageOps.exif_transpose(image).convert(\'RGB\').copy()\n    except (UnidentifiedImageError, OSError, Image.DecompressionBombError,\n            Image.DecompressionBombWarning) as exc:\n        raise ValueError(\'Không đọc được ảnh hoặc ảnh quá lớn. Chọn JPG/PNG khác.\') from exc\n\n\ndef crop_roi(image: Image.Image, roi) -> Image.Image:\n    try:\n        x0, y0, x1, y1 = [float(v) for v in roi]\n    except (TypeError, ValueError) as exc:\n        raise ValueError(\'Cần chọn đủ bốn tọa độ vùng ảnh.\') from exc\n    if not (all(math.isfinite(v) for v in (x0, y0, x1, y1))\n            and 0 <= x0 < x1 <= 1 and 0 <= y0 < y1 <= 1):\n        raise ValueError(\'Vùng chọn phải nằm trong ảnh và có chiều rộng, chiều cao dương.\')\n    box = (math.floor(x0 * image.width), math.floor(y0 * image.height),\n           min(image.width, math.ceil(x1 * image.width)),\n           min(image.height, math.ceil(y1 * image.height)))\n    if box[2] - box[0] < 32 or box[3] - box[1] < 32:\n        raise ValueError(\'Chọn vùng lớn hơn: ảnh cắt cần ít nhất 32 × 32 pixel.\')\n    return image.crop(box)\n\n\ndef image_bytes(image: Image.Image) -> bytes:\n    output = BytesIO()\n    image.convert(\'RGB\').save(output, format=\'PNG\')\n    return output.getvalue()\n\n\ndef preview_url(image: Image.Image) -> str:\n    preview = image.copy()\n    preview.thumbnail((1100, 650))\n    output = BytesIO()\n    preview.save(output, format=\'JPEG\', quality=88)\n    return \'data:image/jpeg;base64,\' + base64.b64encode(output.getvalue()).decode(\'ascii\')\n', 'backend.py': '"""Gradio bridge to the existing notebook runner. Never loads a model."""\n\nimport hashlib\nfrom pathlib import Path\nimport threading\n\ntry:\n    from .images import normalize_image\nexcept ImportError:\n    from images import normalize_image\n\n\nclass ModelAPI:\n    def __init__(self, runner):\n        self.runner = runner\n        self.lock = threading.Lock()\n\n    def status(self):\n        with self.lock:\n            info = self.runner.info()\n        return {name: info[name] for name in (\n            \'model_instance_id\', \'model_load_count\', \'successful_requests\',\n            \'load_seconds\', \'protocol_id\', \'protocol\')}\n\n    def compare(self, before, after):\n        if not before or not after:\n            raise ValueError(\'Cần chọn đủ ảnh trước và ảnh sau.\')\n        data = []\n        for path in (before, after):\n            path = Path(path)\n            if not path.is_file() or path.stat().st_size > 20 * 1024 * 1024:\n                raise ValueError(\'Ảnh không hợp lệ hoặc vượt 20 MB.\')\n            data.append(path.read_bytes())\n        images = [normalize_image(content) for content in data]\n        with self.lock:\n            result = self.runner.compare(*images)\n        return {**result, \'before_sha256\': hashlib.sha256(data[0]).hexdigest(),\n                \'after_sha256\': hashlib.sha256(data[1]).hexdigest()}\n\n\ndef build_demo(runner):\n    import gradio as gr\n    api = ModelAPI(runner)\n    with gr.Blocks(title=\'M3Diff · Kết nối GPU\',\n                   analytics_enabled=False, delete_cache=(3600, 3600)) as app:\n        gr.Markdown(\'## M³Diff · GPU đang sẵn sàng\\n\'\n                    \'Mở giao diện trên laptop để so sánh toàn ảnh hoặc vùng chọn. \'\n                    \'Giữ phiên Colab hoạt động trong lúc sử dụng.\')\n        with gr.Row():\n            before = gr.File(label=\'Ảnh trước\', file_types=[\'.jpg\', \'.jpeg\', \'.png\'], type=\'filepath\')\n            after = gr.File(label=\'Ảnh sau\', file_types=[\'.jpg\', \'.jpeg\', \'.png\'], type=\'filepath\')\n        result = gr.JSON(label=\'Kết quả\')\n        compare = gr.Button(\'So sánh\', variant=\'primary\')\n        compare.click(api.compare, [before, after], result,\n                      api_name=\'compare\', concurrency_id=\'m3diff\', concurrency_limit=1)\n        status_button = gr.Button(\'Kiểm tra kết nối\')\n        status_button.click(api.status, None, result, api_name=\'status\',\n                            concurrency_id=\'m3diff\', concurrency_limit=1)\n    return app.queue(default_concurrency_limit=1, max_size=8)\n'}
web_root = Path(WORK) / 'm3diff_web_bridge'
web_root.mkdir(parents=True, exist_ok=True)
(web_root / '__init__.py').write_text('', encoding='utf-8')
for name, content in WEB_SOURCES.items():
    (web_root / name).write_text(content, encoding='utf-8')
if str(web_root.parent) not in sys.path:
    sys.path.insert(0, str(web_root.parent))
from m3diff_web_bridge.backend import build_demo
print('Sẵn sàng mở kết nối. Model instance:', web_info['model_instance_id'])
print('Lần nạp model trong runner:', web_info['model_load_count'])
print('Gradio:', importlib.metadata.version('gradio'))


In [ ]:
#@title 7. Tải file kết nối — dùng file này trên web laptop
# Ô 7 — mở kết nối có mật khẩu, tải file cấu hình về laptop
import json, secrets
from google.colab import files
if 'build_demo' not in globals():
    raise RuntimeError('Chạy ô 6 trước.')
if 'web_demo' in globals():
    web_demo.close()  # chỉ đóng web cũ, KHÔNG đóng runner/model
if 'web_password' not in globals():
    web_password = secrets.token_urlsafe(18)
web_username = 'm3diff'
web_demo = build_demo(runner)
blocked = [str(globals()[name]) for name in ('MODEL_PATH', 'SOURCE') if name in globals()]
_, local_url, share_url = web_demo.launch(
    share=True, auth=(web_username, web_password), inline=False,
    prevent_thread_lock=True, show_error=True, max_file_size='20mb',
    blocked_paths=blocked)
if not share_url:
    raise RuntimeError('Chưa tạo được link Colab. Giữ model, chạy lại riêng ô 7.')
connection = {'url': share_url, 'username': web_username, 'password': web_password}
connection_path = Path(WORK) / 'm3diff_connection.json'
connection_path.write_text(json.dumps(connection, indent=2), encoding='utf-8')
print('Link:', share_url)
print('Tài khoản:', web_username)
print('Mật khẩu:', web_password)
print('Giữ Colab hoạt động. Mở file kết nối ở thanh bên của web laptop, rồi bấm Kết nối / kiểm tra.')
print('Không cần chạy lại ô nạp model. Không chia sẻ file kết nối hoặc notebook có mật khẩu.')
files.download(str(connection_path))
